# Minimal Self-Contained JAX Flash KMeans (TPU)

This notebook is intentionally self-contained and does **not** import code from this repository.

It includes:
- in-notebook dependency install
- in-notebook experiment config
- synthetic data generation
- flash-style chunked JAX KMeans implementation
- execution diagnostics and quality metrics
- result visualization

## How to run
1. Open in Google Colab.
2. Set runtime to TPU (`Runtime -> Change runtime type -> TPU`).
3. Run all cells top to bottom.

In [ ]:
%pip install -q -U pip
%pip install -q "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
%pip install -q numpy scikit-learn plotly

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, replace
import time
from typing import Any

import jax
import jax.numpy as jnp
import numpy as np
import plotly.express as px

print("JAX version:", jax.__version__)
print("JAX backend:", jax.default_backend())
print("Visible devices:", jax.devices())


def elapsed_ms(start: float) -> float:
    """Return elapsed milliseconds since a perf_counter start value."""
    return (time.perf_counter() - start) * 1000.0


@dataclass(slots=True)
class ExperimentConfig:
    """Single place for all tunable experiment settings."""

    n_samples: int = 20000
    n_features: int = 16
    n_clusters: int = 8
    cluster_std: float = 0.7
    seed: int = 42
    max_iter: int = 100
    tol: float = 1e-4
    n_init: int = 4
    data_chunk_size: int = 4096
    centroid_chunk_size: int = 128


@dataclass(slots=True)
class FlashKMeansResult:
    """Structured fit outputs from flash-style KMeans."""

    centroids: np.ndarray
    labels: np.ndarray
    inertia: float
    n_iter: int
    converged: bool
    fit_time_ms: float


CFG = ExperimentConfig()
CFG

In [ ]:
def validate_config(cfg: ExperimentConfig) -> None:
    """Validate configuration values before data generation and fitting."""
    if cfg.n_samples <= 0:
        raise ValueError("n_samples must be positive.")
    if cfg.n_features <= 0:
        raise ValueError("n_features must be positive.")
    if cfg.n_clusters <= 0:
        raise ValueError("n_clusters must be positive.")
    if cfg.n_clusters > cfg.n_samples:
        raise ValueError("n_clusters must be <= n_samples.")
    if cfg.max_iter <= 0:
        raise ValueError("max_iter must be positive.")
    if cfg.tol < 0:
        raise ValueError("tol must be >= 0.")
    if cfg.n_init <= 0:
        raise ValueError("n_init must be positive.")
    if cfg.data_chunk_size <= 0 or cfg.centroid_chunk_size <= 0:
        raise ValueError("Chunk sizes must be positive.")


def generate_synthetic_data(cfg: ExperimentConfig) -> tuple[np.ndarray, np.ndarray]:
    """Generate deterministic Gaussian-cluster synthetic data without sklearn."""
    validate_config(cfg)
    rng = np.random.default_rng(cfg.seed)

    centers = rng.uniform(-10.0, 10.0, size=(cfg.n_clusters, cfg.n_features)).astype(np.float32)
    labels = rng.integers(0, cfg.n_clusters, size=cfg.n_samples, dtype=np.int32)
    noise = rng.normal(0.0, cfg.cluster_std, size=(cfg.n_samples, cfg.n_features)).astype(np.float32)
    x = centers[labels] + noise
    return x.astype(np.float32), labels.astype(np.int32)


In [ ]:
def assign_labels_chunked(
    x_jax: jax.Array,
    centroids: jax.Array,
    data_chunk_size: int,
    centroid_chunk_size: int,
) -> tuple[jax.Array, jax.Array]:
    """Assign nearest centroids using chunked distance computation."""
    n_samples = int(x_jax.shape[0])
    n_clusters = int(centroids.shape[0])

    all_best_labels = jnp.zeros((n_samples,), dtype=jnp.int32)
    all_best_dist = jnp.full((n_samples,), jnp.inf, dtype=x_jax.dtype)

    for data_start in range(0, n_samples, data_chunk_size):
        data_end = min(data_start + data_chunk_size, n_samples)
        x_chunk = x_jax[data_start:data_end]

        chunk_best_labels = jnp.zeros((data_end - data_start,), dtype=jnp.int32)
        chunk_best_dist = jnp.full((data_end - data_start,), jnp.inf, dtype=x_jax.dtype)

        for centroid_start in range(0, n_clusters, centroid_chunk_size):
            centroid_end = min(centroid_start + centroid_chunk_size, n_clusters)
            c_chunk = centroids[centroid_start:centroid_end]
            dists = jnp.sum((x_chunk[:, None, :] - c_chunk[None, :, :]) ** 2, axis=2)

            local_best_dist = jnp.min(dists, axis=1)
            local_best_labels = jnp.argmin(dists, axis=1).astype(jnp.int32) + centroid_start

            better_mask = local_best_dist < chunk_best_dist
            chunk_best_dist = jnp.where(better_mask, local_best_dist, chunk_best_dist)
            chunk_best_labels = jnp.where(better_mask, local_best_labels, chunk_best_labels)

        all_best_dist = all_best_dist.at[data_start:data_end].set(chunk_best_dist)
        all_best_labels = all_best_labels.at[data_start:data_end].set(chunk_best_labels)

    return all_best_labels, all_best_dist


def update_centroids(
    x_jax: jax.Array,
    labels: jax.Array,
    prev_centroids: jax.Array,
    point_losses: jax.Array,
) -> jax.Array:
    """Update centroids and deterministically reseed empty clusters by highest loss.
    
    - If no cluster is empty, update is standard mean recomputation.
    - If m clusters are empty:
        - rank points by descending loss,
        - take top m points,
        - assign those points as new centroids for the empty cluster indices (in index order).
    """
    n_clusters = int(prev_centroids.shape[0])
    counts = jnp.bincount(labels, length=n_clusters).astype(x_jax.dtype)
    weighted_sums = jax.vmap(
        lambda col: jnp.bincount(labels, weights=col, length=n_clusters),
        in_axes=1,
        out_axes=1,
    )(x_jax)
    new_centroids = weighted_sums / jnp.maximum(counts[:, None], 1.0)

    empty_mask = counts == 0
    n_empty = int(jnp.sum(empty_mask))
    if n_empty == 0:
        return new_centroids

    ranked_idx = jnp.argsort(-point_losses)
    reseed_points = x_jax[ranked_idx[:n_empty]]
    empty_idx = jnp.where(empty_mask)[0]
    return new_centroids.at[empty_idx].set(reseed_points)


def run_single_init(
    x_jax: jax.Array,
    initial_centroids: jax.Array,
    cfg: ExperimentConfig,
) -> tuple[jax.Array, jax.Array, jax.Array, bool, int]:
    """Run one KMeans initialization and return final state."""
    centroids = initial_centroids
    converged = False
    n_iter = 0

    for i in range(cfg.max_iter):
        labels, point_losses = assign_labels_chunked(
            x_jax=x_jax,
            centroids=centroids,
            data_chunk_size=cfg.data_chunk_size,
            centroid_chunk_size=cfg.centroid_chunk_size,
        )
        updated = update_centroids(x_jax, labels, centroids, point_losses)
        """
        Notes:
        Convergence scales with centroid magnitude, so stopping is more stable across differently scaled datasets.
        centroid_norm = max(||c_k||)
        threshold = cfg.tol * (1.0 + centroid_norm)
        stops when shift <= threshold
        """
        shift = float(jnp.max(jnp.linalg.norm(updated - centroids, axis=1)))
        centroid_norm = float(jnp.max(jnp.linalg.norm(centroids, axis=1)))
        threshold = cfg.tol * (1.0 + centroid_norm)
        centroids = updated
        n_iter = i + 1
        if shift <= threshold:
            converged = True
            break

    final_labels, final_dist = assign_labels_chunked(
        x_jax=x_jax,
        centroids=centroids,
        data_chunk_size=cfg.data_chunk_size,
        centroid_chunk_size=cfg.centroid_chunk_size,
    )
    inertia = jnp.sum(final_dist)
    return centroids, final_labels, inertia, converged, n_iter


def init_centroids_kmeanspp(x_jax: jax.Array, n_clusters: int, key: jax.Array) -> jax.Array:
    """Initialize centroids with k-means++ sampling.
    
    - picks first centroid uniformly at random
    - iteratively samples next centroids proportional to squared distance to nearest selected centroid
    - updates nearest-distance cache each step
    - includes a safe fallback to uniform-over-unselected if distance mass collapses to zero
    
    """
    n_samples = int(x_jax.shape[0])

    key, first_key = jax.random.split(key)
    first_idx = jax.random.randint(first_key, shape=(), minval=0, maxval=n_samples)

    selected = jnp.zeros((n_samples,), dtype=bool)
    selected = selected.at[first_idx].set(True)

    first_centroid = x_jax[first_idx]
    min_dist = jnp.sum((x_jax - first_centroid[None, :]) ** 2, axis=1)

    chosen_indices: list[jax.Array] = [first_idx]

    for _ in range(1, n_clusters):
        weights = jnp.where(selected, 0.0, min_dist)
        weight_sum = float(jnp.sum(weights))

        if weight_sum > 0.0:
            probs = weights / jnp.sum(weights)
        else:
            candidates = jnp.where(selected, 0.0, 1.0)
            probs = candidates / jnp.sum(candidates)

        key, pick_key = jax.random.split(key)
        next_idx = jax.random.choice(pick_key, n_samples, shape=(), replace=False, p=probs)

        selected = selected.at[next_idx].set(True)
        next_centroid = x_jax[next_idx]
        next_dist = jnp.sum((x_jax - next_centroid[None, :]) ** 2, axis=1)
        min_dist = jnp.minimum(min_dist, next_dist)

        chosen_indices.append(next_idx)

    init_idx = jnp.stack(chosen_indices).astype(jnp.int32)
    return x_jax[init_idx]


def fit_flash_kmeans(x: np.ndarray, cfg: ExperimentConfig) -> FlashKMeansResult:
    """Fit flash-style KMeans with multi-init selection by best inertia."""
    validate_config(cfg)
    x_np = np.asarray(x, dtype=np.float32)
    x_jax = jnp.asarray(x_np)

    key = jax.random.PRNGKey(cfg.seed)
    best: tuple[jax.Array, jax.Array, jax.Array, bool, int] | None = None
    best_inertia = float("inf")

    start = time.perf_counter()
    for _ in range(cfg.n_init):
        key, subkey = jax.random.split(key)
        initial_centroids = init_centroids_kmeanspp(
            x_jax=x_jax,
            n_clusters=cfg.n_clusters,
            key=subkey,
        )

        out = run_single_init(x_jax=x_jax, initial_centroids=initial_centroids, cfg=cfg)
        inertia_value = float(out[2])
        if inertia_value < best_inertia:
            best_inertia = inertia_value
            best = out

    fit_time = elapsed_ms(start)
    if best is None:
        raise RuntimeError("No KMeans result was produced.")

    centroids, labels, inertia, converged, n_iter = best
    return FlashKMeansResult(
        centroids=np.asarray(centroids, dtype=np.float32),
        labels=np.asarray(labels, dtype=np.int32),
        inertia=float(inertia),
        n_iter=n_iter,
        converged=bool(converged),
        fit_time_ms=fit_time,
    )

In [ ]:
def compute_quality_metrics(x: np.ndarray, labels: np.ndarray) -> dict[str, float | None]:
    """Compute optional clustering quality metrics with safe fallbacks."""
    unique_labels = np.unique(labels)
    if unique_labels.size < 2 or unique_labels.size >= x.shape[0]:
        return {"silhouette": None, "calinski_harabasz": None, "davies_bouldin": None}

    try:
        from sklearn.metrics import (
            calinski_harabasz_score,
            davies_bouldin_score,
            silhouette_score,
        )
    except Exception:
        return {"silhouette": None, "calinski_harabasz": None, "davies_bouldin": None}

    metrics: dict[str, float | None] = {}
    for name, fn in {
        "silhouette": silhouette_score,
        "calinski_harabasz": calinski_harabasz_score,
        "davies_bouldin": davies_bouldin_score,
    }.items():
        try:
            metrics[name] = float(fn(x, labels))
        except Exception:
            metrics[name] = None
    return metrics


def project_to_2d(x: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Project points to 2D and return projection with center and basis."""
    x = np.asarray(x, dtype=np.float32)
    center = x.mean(axis=0, keepdims=True)
    x_centered = x - center

    if x.shape[1] <= 2:
        basis = np.eye(x.shape[1], dtype=np.float32)
        x2 = x_centered[:, :2]
        if x2.shape[1] == 1:
            x2 = np.concatenate([x2, np.zeros((x2.shape[0], 1), dtype=np.float32)], axis=1)
        return x2, center, basis

    _, _, vt = np.linalg.svd(x_centered, full_matrices=False)
    basis = vt[:2].T.astype(np.float32)
    return x_centered @ basis, center, basis


def apply_projection(x: np.ndarray, center: np.ndarray, basis: np.ndarray) -> np.ndarray:
    """Apply a precomputed 2D projection basis to another matrix."""
    x_centered = np.asarray(x, dtype=np.float32) - center
    if basis.shape[1] == 1:
        x2 = x_centered[:, :1]
        return np.concatenate([x2, np.zeros((x2.shape[0], 1), dtype=np.float32)], axis=1)
    return x_centered @ basis


def plot_clusters_2d(x: np.ndarray, labels: np.ndarray, centroids: np.ndarray, title: str) -> Any:
    """Create an interactive 2D Plotly scatter for points and centroids."""
    x2, center, basis = project_to_2d(x)
    c2 = apply_projection(centroids, center, basis)

    fig = px.scatter(
        x=x2[:, 0],
        y=x2[:, 1],
        color=labels.astype(str),
        opacity=0.55,
        title=title,
        labels={"x": "component_1", "y": "component_2", "color": "cluster"},
    )
    fig.add_scatter(
        x=c2[:, 0],
        y=c2[:, 1],
        mode="markers",
        marker={"size": 12, "symbol": "x", "line": {"width": 2}},
        name="centroids",
    )
    return fig


def plot_true_vs_pred_clusters(
    x: np.ndarray,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    centroids: np.ndarray,
    title: str,
) -> Any:
    """Plot true vs predicted clusters with per-cluster legend toggles."""
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    x2, center, basis = project_to_2d(x)
    c2 = apply_projection(centroids, center, basis)

    clusters = np.unique(np.concatenate([y_true.astype(np.int32), y_pred.astype(np.int32)]))
    palette = px.colors.qualitative.Alphabet

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Ground Truth Clusters", "Predicted Clusters"),
        horizontal_spacing=0.08,
    )

    for cluster_id in clusters:
        color = palette[int(cluster_id) % len(palette)]
        true_mask = y_true == cluster_id
        pred_mask = y_pred == cluster_id

        fig.add_trace(
            go.Scattergl(
                x=x2[true_mask, 0],
                y=x2[true_mask, 1],
                mode="markers",
                marker={"size": 5, "opacity": 0.6, "color": color},
                name=f"cluster {int(cluster_id)}",
                legendgroup=f"cluster_{int(cluster_id)}",
                showlegend=True,
            ),
            row=1,
            col=1,
        )

        fig.add_trace(
            go.Scattergl(
                x=x2[pred_mask, 0],
                y=x2[pred_mask, 1],
                mode="markers",
                marker={"size": 5, "opacity": 0.6, "color": color},
                name=f"cluster {int(cluster_id)}",
                legendgroup=f"cluster_{int(cluster_id)}",
                showlegend=False,
            ),
            row=1,
            col=2,
        )

    fig.add_trace(
        go.Scattergl(
            x=c2[:, 0],
            y=c2[:, 1],
            mode="markers",
            marker={"size": 11, "symbol": "x", "color": "black", "line": {"width": 2}},
            name="pred centroids",
            legendgroup="pred_centroids",
            showlegend=True,
        ),
        row=1,
        col=2,
    )

    fig.update_layout(
        title=title,
        legend={"groupclick": "togglegroup"},
        height=560,
        width=1200,
    )
    fig.update_xaxes(title_text="component_1", row=1, col=1)
    fig.update_yaxes(title_text="component_2", row=1, col=1)
    fig.update_xaxes(title_text="component_1", row=1, col=2)
    fig.update_yaxes(title_text="component_2", row=1, col=2)
    return fig

In [ ]:
def run_experiment(cfg: ExperimentConfig) -> dict[str, Any]:
    """Run synthetic generation, fitting, metrics, and summary collection."""
    total_start = time.perf_counter()
    x, y_true = generate_synthetic_data(cfg)
    fit_result = fit_flash_kmeans(x, cfg)
    metrics = compute_quality_metrics(x, fit_result.labels)

    cluster_sizes = np.bincount(fit_result.labels, minlength=cfg.n_clusters)
    total_runtime_ms = elapsed_ms(total_start)

    return {
        "x": x,
        "y_true": y_true,
        "fit": fit_result,
        "metrics": metrics,
        "cluster_sizes": cluster_sizes,
        "total_runtime_ms": total_runtime_ms,
    }


result = run_experiment(CFG)
fit = result["fit"]

print("Converged:", fit.converged)
print("Iterations:", fit.n_iter)
print("Inertia:", round(fit.inertia, 4))
print("Fit time (ms):", round(fit.fit_time_ms, 2))
print("Total runtime (ms):", round(result["total_runtime_ms"], 2))
print("Cluster sizes:", result["cluster_sizes"])
print("Quality metrics:", result["metrics"])

fig_pred = plot_clusters_2d(
    x=result["x"],
    labels=fit.labels,
    centroids=fit.centroids,
    title="Minimal JAX Flash KMeans Result",
)
fig_pred.show()

In [ ]:
fig_compare_true_vs_pred = plot_true_vs_pred_clusters(
    x=result["x"],
    y_true=result["y_true"],
    y_pred=fit.labels,
    centroids=fit.centroids,
    title="Ground Truth vs Predicted Clusters (toggle legend per cluster)",
)
fig_compare_true_vs_pred.show()

In [ ]:
def sweep_data_chunk_sizes(cfg: ExperimentConfig, x: np.ndarray, chunk_sizes: list[int]) -> Any:
    """Measure fit-time trend across data chunk sizes for quick tradeoff checks."""
    rows: list[dict[str, float]] = []
    for size in chunk_sizes:
        test_cfg = replace(cfg, data_chunk_size=int(size))
        fit = fit_flash_kmeans(x, test_cfg)
        rows.append({"data_chunk_size": float(size), "fit_time_ms": fit.fit_time_ms})

    fig = px.line(
        rows,
        x="data_chunk_size",
        y="fit_time_ms",
        markers=True,
        title="Chunk Size vs Fit Time",
    )
    return fig


chunk_fig = sweep_data_chunk_sizes(CFG, result["x"], [1024, 2048, 4096, 8192])
chunk_fig.show()